# 6a. Second-annotator tool (double annotation)

| | |
|---|---|
| **Stage** | 6a of 9 |
| **Purpose** | Interactive tool used by the second annotator to label the double-annotation sample blind to the gold labels: 60 random sentences, then a 20-sentence top-up that adds examples of rare aspects. Progress is saved after every sentence. |
| **Inputs** | `outputs/supplementary/iaa_blind_for_annotator2.csv`<br>`outputs/supplementary/iaa_blind_topup_for_annotator2.csv`<br>`scripts/aspect_taxonomy.py` |
| **Outputs** | `outputs/supplementary/iaa_annotator2.csv`<br>`outputs/supplementary/iaa_annotator2_topup.csv` |
| **Runtime** | CPU only. Interactive (ipywidgets); 80 sentences. |
| **Depends on** | Stages 2 and 5 (the blind samples and their keys are created in notebook 5, sections S3a and S3a-2) |
| **Configuration** | Set `PROJECT_ROOT` in the first code cell if your Drive folder is not `MyDrive/GameSense`. |

## Instructions for the second annotator

1. A CPU runtime is enough (no GPU needed).
2. Make sure the shared `GameSense` folder has been added to your My Drive (in Drive: right-click the folder > Organize > Add shortcut > My Drive) and that you have editor permission.
3. Run the cell below. There are 80 sentences: Part 1 has 60 sentences; when it is finished the tool moves on to the 20 sentences of Part 2.
4. For each sentence, choose the author's attitude towards every listed aspect; keep `— not mentioned` if the aspect is not mentioned. Click **Save & next** for every sentence, including sentences that mention none of the aspects.
5. Progress is saved automatically. If you close the notebook, rerun the cell to continue where you left off.
6. While annotating, do not open any file in `outputs/supplementary/` whose name contains `key` or `gold`.

In [ ]:
# ===== IAA annotation tool for annotator 2: ONE cell, runs Part 1 (60 sentences) then Part 2 (20 sentences) =====
import os, sys, html
import pandas as pd
import ipywidgets as widgets
from IPython.display import display

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    pass

# ---------- Settings ----------
PROJECT_ROOT = "/content/drive/MyDrive/GameSense"       # the shared GameSense folder (added as a shortcut to My Drive)
SUPP = f"{PROJECT_ROOT}/outputs/supplementary"
PARTS = [("Part 1", "iaa_blind_for_annotator2.csv",       "iaa_annotator2.csv"),
         ("Part 2", "iaa_blind_topup_for_annotator2.csv", "iaa_annotator2_topup.csv")]
NONE_OPT  = "— not mentioned"
OPTIONS   = [NONE_OPT, "negative", "neutral", "positive"]
GAME_NAME = {"royalmatch": "Royal Match", "genshin": "Genshin Impact", "mlbb": "Mobile Legends",
             "pubgm": "PUBG Mobile", "arknights": "Arknights"}

def load_guide():
    """Aspect labels/descriptions from the official taxonomy file, so both annotators see the same definitions."""
    try:
        sys.path.insert(0, f"{PROJECT_ROOT}/scripts")
        from aspect_taxonomy import ASPECT_TAXONOMY
        return {a: (str(m["label"]), str(m["description"])) for a, m in ASPECT_TAXONOMY.items()}
    except Exception as e:
        print("WARNING: could not load aspect_taxonomy.py; only aspect ids will be shown:", e)
        return {}

def load_part(name, blind, work):
    bp, wp = f"{SUPP}/{blind}", f"{SUPP}/{work}"
    if not os.path.exists(wp) and not os.path.exists(bp):
        raise FileNotFoundError(f"Cannot find {bp}\nCheck PROJECT_ROOT, and that the shared GameSense folder "
                                f"was added to 'My Drive' (right-click the folder > Organize > Add shortcut).")
    if os.path.exists(wp):                              # resume
        df = pd.read_csv(wp)
    else:
        df = pd.read_csv(bp)
        df["_done"] = 0
        df.to_csv(wp, index=False)
    ann_cols = [c for c in df.columns if c.startswith("ann_")]
    df[ann_cols] = df[ann_cols].fillna("").astype(str)
    df["_done"] = pd.to_numeric(df["_done"], errors="coerce").fillna(0).astype(int)
    return {"name": name, "df": df, "work": wp}

def annotate_iaa():
    guide = load_guide()
    parts = [load_part(*p) for p in PARTS]

    def first_pending(p):
        idx = parts[p]["df"].index[parts[p]["df"]["_done"] == 0].tolist()
        return idx[0] if idx else None

    # Start at the first part that still has unconfirmed sentences
    state = {"p": len(parts), "i": 0, "dds": {}}
    for p in range(len(parts)):
        if first_pending(p) is not None:
            state["p"], state["i"] = p, first_pending(p)
            break

    head, sent, rows_box, msg = widgets.HTML(), widgets.HTML(), widgets.VBox([]), widgets.HTML()
    rules = widgets.HTML(
        "<div style='font-size:12px;color:#999;margin-top:10px;line-height:1.6'>"
        "<b>For each listed aspect: choose the sentiment toward THAT aspect, or leave '— not mentioned'.</b><br>"
        "&bull; Pure emotional expressions (e.g. 'very relaxing', 'so much fun') &rarr; all '— not mentioned'.<br>"
        "&bull; Feature suggestions and truncated fragments &rarr; all '— not mentioned'.<br>"
        "&bull; 'gacha rates terrible, spent so much' &rarr; gacha_rates only (not monetization_value, unless explicit pricing language).<br>"
        "&bull; A sentence may mention several aspects with different sentiments.<br>"
        "&bull; Press <b>Save &amp; next</b> to confirm each sentence (even if nothing is mentioned)."
        "</div>")

    def total_done():
        return sum(int(pt["df"]["_done"].sum()) for pt in parts), sum(len(pt["df"]) for pt in parts)

    def show():
        msg.value = ""
        done, total = total_done()
        p, i = state["p"], state["i"]
        if p >= len(parts):
            head.value = f"<h3>All done ({done}/{total}). Thank you!</h3>"
            sent.value, rows_box.children, state["dds"] = "", [], {}
            return
        pt, df = parts[p], parts[p]["df"]
        r = df.loc[i]
        head.value = (f"<b>{pt['name']} of {len(parts)} &middot; Sentence {i + 1}/{len(df)}</b> &nbsp; "
                      f"<span style='color:#999'>confirmed overall: {done}/{total}</span>")
        sent.value = (f"<p style='font-size:19px;margin-bottom:2px'>{html.escape(str(r['sentence_text']))}</p>"
                      f"<p style='font-size:12px;color:#8fd0ff'>Game: {html.escape(GAME_NAME.get(r['game'], str(r['game'])))}</p>")
        state["dds"], rows = {}, []
        for a in [x.strip() for x in str(r["applicable_aspects"]).split(",") if x.strip()]:
            label, desc = guide.get(a, (a, ""))
            saved = df.at[i, f"ann_{a}"] if f"ann_{a}" in df.columns else ""
            dd = widgets.Dropdown(options=OPTIONS, value=saved if saved in OPTIONS else NONE_OPT,
                                  layout=widgets.Layout(width="190px"))
            state["dds"][a] = dd
            info = widgets.HTML(f"<b>{html.escape(label)}</b><br>"
                                f"<span style='font-size:11px;color:#999'>{html.escape(desc)}</span>",
                                layout=widgets.Layout(width="560px"))
            rows.append(widgets.HBox([info, dd], layout=widgets.Layout(margin="4px 0")))
        rows_box.children = rows

    def save_and_next(_):
        p, i = state["p"], state["i"]
        if p >= len(parts):
            return
        pt, df = parts[p], parts[p]["df"]
        for a, dd in state["dds"].items():
            df.at[i, f"ann_{a}"] = "" if dd.value == NONE_OPT else dd.value
        df.at[i, "_done"] = 1
        df.to_csv(pt["work"], index=False)              # auto-save into the shared folder
        if i + 1 < len(df):                             # next sentence in this part
            state["i"] = i + 1
        else:                                           # move on to the next part (first unconfirmed sentence)
            state["p"], state["i"] = p + 1, 0
            if state["p"] < len(parts):
                fp = first_pending(state["p"])
                state["i"] = fp if fp is not None else 0
        show()

    def back(_):
        p, i = state["p"], state["i"]
        if p >= len(parts):                             # from the finished screen: go to the very last sentence
            state["p"], state["i"] = len(parts) - 1, len(parts[-1]["df"]) - 1
        elif i > 0:
            state["i"] = i - 1
        elif p > 0:
            state["p"], state["i"] = p - 1, len(parts[p - 1]["df"]) - 1
        else:
            return
        show()
        msg.value = "<span style='color:#999'>Editing a previous sentence: press Save &amp; next to confirm changes.</span>"

    b_next = widgets.Button(description="Save & next", button_style="success", layout=widgets.Layout(width="160px"))
    b_back = widgets.Button(description="Back", layout=widgets.Layout(width="100px"))
    b_next.on_click(save_and_next)
    b_back.on_click(back)

    show()
    display(widgets.VBox([head, sent, rows_box, widgets.HBox([b_back, b_next]), msg, rules]))
    return parts, state, b_next, b_back      # returned for testing only; the call below discards it

_ = annotate_iaa()